In [1]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
from nn_utils import split_and_shuffle, MLP, train_nn, load_nn, evaluate_nn,train_two_branch, TwoBranchMLP, evaluate_two_branch
import os
from global_vars import *


train_folder = os.path.join(data_folder, 'nodeep_train_data_ver1/')
test_folder = os.path.join(data_folder, 'nodeep_test_data_ver1/')


X_train_val = pd.read_parquet(os.path.join(train_folder, "X_train.parquet"),engine="fastparquet")
y_t_train_val = pd.read_parquet(os.path.join(train_folder, "y_t_train.parquet"),engine="fastparquet")
y_q_train_val = pd.read_parquet(os.path.join(train_folder, "y_q_train.parquet"),engine="fastparquet")

X_test = pd.read_parquet(os.path.join(test_folder, "X_test.parquet"),engine="fastparquet")
y_t_test = pd.read_parquet(os.path.join(test_folder, "y_t_test.parquet"),engine="fastparquet")
y_q_test = pd.read_parquet(os.path.join(test_folder, "y_q_test.parquet"),engine="fastparquet")

In [2]:
X_train, y_train, X_val, y_val = split_and_shuffle(
    X_train_val.values, y_t_train_val.values, val_frac=0.2, seed=0
)

In [3]:
n_surf_features = len(surf_input_var_names)
n_total_features = X_train.shape[1]
branch2_start = (n_total_features-n_surf_features)
Xa_train = X_train[:, range(branch2_start)]
Xb_train = X_train[:, range(branch2_start, n_total_features)]
Xa_val = X_val[:, range(branch2_start)]
Xb_val = X_val[:, range(branch2_start, n_total_features)]

In [22]:
model, scaler, history = train_two_branch(
    Xa_train, Xb_train, y_train,
    branch_args={
        "branch_a_hidden": (128,128,),
        "branch_b_hidden": (64,),
        "head_hidden":     (128,64),
    },
    epochs=200,
    Xa_val=Xa_val, Xb_val=Xb_val, y_val=y_val,
    patience=15,
)

epoch   0 | train 0.6180 | val 0.4727
epoch   1 | train 0.5495 | val 0.4361
epoch   2 | train 0.5282 | val 0.4341
epoch   3 | train 0.5084 | val 0.4267
epoch   4 | train 0.4981 | val 0.4079
epoch   5 | train 0.4910 | val 0.4124
epoch   6 | train 0.4801 | val 0.3978
epoch   7 | train 0.4744 | val 0.4157
epoch   8 | train 0.4689 | val 0.3902
epoch   9 | train 0.4612 | val 0.3868
epoch  10 | train 0.4574 | val 0.3996
epoch  11 | train 0.4526 | val 0.4065
epoch  12 | train 0.4457 | val 0.3770
epoch  13 | train 0.4497 | val 0.3676
epoch  14 | train 0.4406 | val 0.3715
epoch  15 | train 0.4416 | val 0.3799
epoch  16 | train 0.4366 | val 0.3813
epoch  17 | train 0.4348 | val 0.3819
epoch  18 | train 0.4326 | val 0.3643
epoch  19 | train 0.4312 | val 0.3680
epoch  20 | train 0.4243 | val 0.3626
epoch  21 | train 0.4298 | val 0.3580
epoch  22 | train 0.4225 | val 0.3715
epoch  23 | train 0.4254 | val 0.3595
epoch  24 | train 0.4218 | val 0.3549
epoch  25 | train 0.4172 | val 0.3674
epoch  26 | 

In [23]:
Xa_test = X_test.values[:, range(branch2_start)]
Xb_test = X_test.values[:, range(branch2_start, n_total_features)]

In [24]:
metrics = evaluate_two_branch(model, scaler, Xa_test, Xb_test, y_t_test)
print(f"RMSE {metrics['rmse']:.4e} | MAE {metrics['mae']:.4e} | R² {metrics['r2_overall']:.4f}")

RMSE 3.4600e-05 | MAE 1.1150e-05 | R² 0.5516


In [5]:
list1 = [
    (128,128,),
    (256,128,)
]
list2 = [
    #(64,),
    (64,32)
]


list3=[
    (128,64),
    (64,32)
]

for a in list1:
    for b in list2:
        for c in list3:

            model, scaler, history = train_two_branch(
                Xa_train, Xb_train, y_train,
                branch_args={
                    "branch_a_hidden": a,
                    "branch_b_hidden": b,
                    "head_hidden":     c,
                },
                epochs=200,
                Xa_val=Xa_val, Xb_val=Xb_val, y_val=y_val,
                patience=15,
            )

epoch   0 | train 0.6204 | val 0.4655
epoch   1 | train 0.5519 | val 0.4587
epoch   2 | train 0.5340 | val 0.4333
epoch   3 | train 0.5137 | val 0.4283
epoch   4 | train 0.4998 | val 0.4113
epoch   5 | train 0.4878 | val 0.4139
epoch   6 | train 0.4806 | val 0.3977
epoch   7 | train 0.4758 | val 0.4075
epoch   8 | train 0.4671 | val 0.3880
epoch   9 | train 0.4603 | val 0.3850
epoch  10 | train 0.4602 | val 0.3800
epoch  11 | train 0.4545 | val 0.4457
epoch  12 | train 0.4503 | val 0.3879
epoch  13 | train 0.4431 | val 0.3735
epoch  14 | train 0.4404 | val 0.4134
epoch  15 | train 0.4361 | val 0.3778
epoch  16 | train 0.4374 | val 0.3701
epoch  17 | train 0.4321 | val 0.3674
epoch  18 | train 0.4285 | val 0.3703
epoch  19 | train 0.4287 | val 0.3621
epoch  20 | train 0.4271 | val 0.3595
epoch  21 | train 0.4240 | val 0.3645
epoch  22 | train 0.4211 | val 0.3679
epoch  23 | train 0.4284 | val 0.3614
epoch  24 | train 0.4224 | val 0.3585
epoch  25 | train 0.4185 | val 0.3625
epoch  26 | 

In [5]:
sizes_lst = [
    (512, 256, 128),
    (512, 256, 128),
    (512, 256, 128),
    (1024, 512, 256),
    (1024, 512, 256),
    (1024, 512, 256),
    (256, 128, 64),
    (256, 128, 64),
    (256, 128, 64),
    (128, 64, 32),
    (128, 64, 32),
    (128, 64, 32), # 12
    (256, 128, 64, 32),
    (256, 128, 64, 32),
    (256, 128, 64, 32),
    (128, 64, 32, 16),
    (128, 64, 32, 16),
    (128, 64, 32, 16),
]

dropouts_lst = [
    [0.2, 0.2, 0.1],
    [0.3, 0.3, 0.1],
    [0.1, 0.1, 0.1],
    [0.2, 0.2, 0.1],
    [0.3, 0.3, 0.1],
    [0.1, 0.1, 0.1],
    [0.2, 0.2, 0.1],
    [0.3, 0.3, 0.1],
    [0.1, 0.1, 0.1],
    [0.2, 0.2, 0.1],
    [0.3, 0.3, 0.1],
    [0.1, 0.1, 0.1], #12
    [0.2, 0.2, 0.1, 0.1],
    [0.3, 0.3, 0.1, 0.1],
    [0.1, 0.1, 0.1, 0.1],
    [0.2, 0.2, 0.1, 0.1],
    [0.3, 0.3, 0.1, 0.1],
    [0.1, 0.1, 0.1, 0.1],
]

for sizes, dropouts in zip(sizes_lst, dropouts_lst):
    print(sizes, dropouts)
    model, scaler, history = train_nn(
        X_train, y_train,
        epochs=200,
        mlp_args={"hidden_sizes": sizes, "dropout": dropouts},
        lr=1e-3,
        X_val=X_val, y_val=y_val,
        patience=20
    )

    metrics = evaluate_nn(model, scaler, X_test.values, y_t_test.values)
    print(f"RMSE {metrics['rmse']:.4e} | MAE {metrics['mae']:.4e} | R² {metrics['r2_overall']:.4f}")

(512, 256, 128) [0.2, 0.2, 0.1]
epoch   0 | train 6.5667e-01 | val 5.1064e-01
epoch   1 | train 6.0339e-01 | val 5.0259e-01
epoch   2 | train 5.8506e-01 | val 5.1622e-01
epoch   3 | train 5.7825e-01 | val 4.4717e-01
epoch   4 | train 5.7400e-01 | val 4.5489e-01
epoch   5 | train 5.6260e-01 | val 4.2900e-01
epoch   6 | train 5.5767e-01 | val 4.3046e-01
epoch   7 | train 5.5424e-01 | val 4.2934e-01
epoch   8 | train 5.4906e-01 | val 4.3017e-01
epoch   9 | train 5.4483e-01 | val 4.3130e-01
epoch  10 | train 5.4421e-01 | val 4.4579e-01
epoch  11 | train 5.3972e-01 | val 4.1844e-01
epoch  12 | train 5.3970e-01 | val 4.1881e-01
epoch  13 | train 5.3517e-01 | val 4.1869e-01
epoch  14 | train 5.3617e-01 | val 4.5191e-01
epoch  15 | train 5.3251e-01 | val 4.1331e-01
epoch  16 | train 5.3116e-01 | val 4.0693e-01
epoch  17 | train 5.2943e-01 | val 4.0672e-01
epoch  18 | train 5.2936e-01 | val 4.0826e-01
epoch  19 | train 5.2715e-01 | val 3.9965e-01
epoch  20 | train 5.2434e-01 | val 4.0327e-01
ep

In [ ]:
metrics = evaluate_nn(model, scaler, X_test.values, y_t_test.values)
print(f"RMSE {metrics['rmse']:.4e} | MAE {metrics['mae']:.4e} | R² {metrics['r2_overall']:.4f}")

In [ ]:
print(torch.__version__)          # a "+cpu" suffix here is the giveaway
print(torch.version.cuda) 